In [1]:
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import streamlit as st
from scipy.interpolate import CubicSpline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeRegressor
from xgboost import XGBRegressor

In [2]:
st.set_page_config(
    page_title="Thermal Properties & Models Analysis App",
    page_icon="📊",
    layout="wide")

st.title("📊 Thermal Properties Analysis & Critical Point Prediction Dashboard")
st.markdown("This application trains machine learning models, evaluates their performance, and plots P-T Phase Envelopes.")
st.sidebar.header("Execution Settings")
csv_file_path = st.sidebar.text_input("CSV File Path", value=r"c:\Users\hp\Desktop\critprop_merged_final.csv")
run_button = st.sidebar.button("Start Training & Analyze Results", type="primary")
if run_button:
    if not os.path.exists(csv_file_path):
        st.error(f"File not found at the specified path: {csv_file_path}")
    else:
        with st.spinner("Reading data and training models, please wait..."):
            os.makedirs("models_output_plots", exist_ok=True)
            os.makedirs("loss_curves_plots", exist_ok=True)
            os.makedirs("envelopes_diagrams/pure", exist_ok=True)
            os.makedirs("envelopes_diagrams/ternary", exist_ok=True)
            os.makedirs("envelopes_diagrams/pentary", exist_ok=True)
            os.makedirs("envelopes_diagrams/combined", exist_ok=True)

            df = pd.read_csv(csv_file_path)
            clean_df = df.dropna(subset=["Vc", "Mw", "Zc", "Tc mean (K)", "Pc mean (bar)"])
            
            train_df, test_df = train_test_split(clean_df, test_size=0.2, random_state=42)
            X_train_raw, y_train_tc, y_train_pc = (
                train_df[["Vc", "Mw", "Zc"]],
                train_df["Tc mean (K)"],
                train_df["Pc mean (bar)"])
            X_test_raw, y_test_tc, y_test_pc = (
                test_df[["Vc", "Mw", "Zc"]],
                test_df["Tc mean (K)"],
                test_df["Pc mean (bar)"])
            
            scaler_X = StandardScaler()
            X_train_scaled = scaler_X.fit_transform(X_train_raw)
            X_test_scaled = scaler_X.transform(X_test_raw)
            
            models_list = ["random_forest", "decision_tree", "mlp", "xgboost"]
            metrics_results = []
            predictions_per_model = []
            model_predictions_dict = {}
            model_colors = {
                "real_data": "#212529",    
                "random_forest": "#1b40ff",
                "decision_tree": "#28a745",
                "mlp": "#ff8c00",
                "xgboost": "#dc3545"}
            
            fig_orig, axes = plt.subplots(1, 2, figsize=(12, 5), dpi=150, facecolor="white")
            axes[0].scatter(clean_df["Vc"], clean_df["Tc mean (K)"], color="purple", alpha=0.6, edgecolors="k")
            axes[0].set_title("Original Data: Vc vs Tc", fontsize=11, fontweight="bold")
            axes[0].set_xlabel("Critical Volume (Vc)")
            axes[0].set_ylabel("Critical Temperature (Tc) [K]")
            axes[0].grid(True, linestyle="--", alpha=0.5)
            
            axes[1].scatter(clean_df["Vc"], clean_df["Pc mean (bar)"], color="teal", alpha=0.6, edgecolors="k")
            axes[1].set_title("Original Data: Vc vs Pc", fontsize=11, fontweight="bold")
            axes[1].set_xlabel("Critical Volume (Vc)")
            axes[1].set_ylabel("Critical Pressure (Pc) [bar]")
            axes[1].grid(True, linestyle="--", alpha=0.5)
            plt.tight_layout()
            
            st.subheader("📈 Original Data Distribution")
            st.pyplot(fig_orig)
            plt.close(fig_orig)
            
            trained_models = {}
            for m_name in models_list:
                X_tr = X_train_scaled if m_name == "mlp" else X_train_raw
                X_te = X_test_scaled if m_name == "mlp" else X_test_raw
                
                if m_name == "random_forest":
                    model_tc = RandomForestRegressor(n_estimators=100, max_depth=8, min_samples_split=10, min_samples_leaf=4, max_features=0.8, random_state=42, warm_start=True)
                    model_pc = RandomForestRegressor(n_estimators=100, max_depth=8, min_samples_split=10, min_samples_leaf=4, max_features=0.8, random_state=42, warm_start=True)
                elif m_name == "decision_tree":
                    model_tc = DecisionTreeRegressor(max_depth=5, min_samples_split=15, min_samples_leaf=8, random_state=42)
                    model_pc = DecisionTreeRegressor(max_depth=5, min_samples_split=15, min_samples_leaf=8, random_state=42)
                elif m_name == "mlp":
                    model_tc = MLPRegressor(hidden_layer_sizes=(64, 32), activation="relu", alpha=0.05, max_iter=1, warm_start=True, random_state=42)
                    model_pc = MLPRegressor(hidden_layer_sizes=(64, 32), activation="relu", alpha=0.05, max_iter=1, warm_start=True, random_state=42)
                elif m_name == "xgboost":
                    model_tc = XGBRegressor(n_estimators=100, learning_rate=0.03, max_depth=4, subsample=0.8, colsample_bytree=0.8, reg_alpha=0.1, reg_lambda=1.0, random_state=42)
                    model_pc = XGBRegressor(n_estimators=100, learning_rate=0.03, max_depth=4, subsample=0.8, colsample_bytree=0.8, reg_alpha=0.1, reg_lambda=1.0, random_state=42)
                    
                train_losses_tc, test_losses_tc = [], []
                train_losses_pc, test_losses_pc = [], []
                
                if m_name == "mlp":
                    for epoch in range(100):
                        model_tc.fit(X_tr, y_train_tc)
                        model_pc.fit(X_tr, y_train_pc)
                        train_losses_tc.append(mean_squared_error(y_train_tc, model_tc.predict(X_tr)))
                        test_losses_tc.append(mean_squared_error(y_test_tc, model_tc.predict(X_te)))
                        train_losses_pc.append(mean_squared_error(y_train_pc, model_pc.predict(X_tr)))
                        test_losses_pc.append(mean_squared_error(y_test_pc, model_pc.predict(X_te)))
                elif m_name in ["random_forest", "xgboost"]:
                    for s in range(5, 105, 5):
                        model_tc.n_estimators = s
                        model_pc.n_estimators = s
                        model_tc.fit(X_tr, y_train_tc)
                        model_pc.fit(X_tr, y_train_pc)
                        train_losses_tc.append(mean_squared_error(y_train_tc, model_tc.predict(X_tr)))
                        test_losses_tc.append(mean_squared_error(y_test_tc, model_tc.predict(X_te)))
                        train_losses_pc.append(mean_squared_error(y_train_pc, model_pc.predict(X_tr)))
                        test_losses_pc.append(mean_squared_error(y_test_pc, model_pc.predict(X_te)))
                else:
                    for d in range(1, 6):
                        model_tc = DecisionTreeRegressor(max_depth=d, min_samples_split=15, min_samples_leaf=8, random_state=42).fit(X_tr, y_train_tc)
                        model_pc = DecisionTreeRegressor(max_depth=d, min_samples_split=15, min_samples_leaf=8, random_state=42).fit(X_tr, y_train_pc)
                        train_losses_tc.append(mean_squared_error(y_train_tc, model_tc.predict(X_tr)))
                        test_losses_tc.append(mean_squared_error(y_test_tc, model_tc.predict(X_te)))
                        train_losses_pc.append(mean_squared_error(y_train_pc, model_pc.predict(X_tr)))
                        test_losses_pc.append(mean_squared_error(y_test_pc, model_pc.predict(X_te)))
                        
                pred_train_tc = model_tc.predict(X_tr)
                pred_test_tc = model_tc.predict(X_te)
                pred_train_pc = model_pc.predict(X_tr)
                pred_test_pc = model_pc.predict(X_te)
                trained_models[m_name] = {"model_tc": model_tc, "model_pc": model_pc}
                
                metrics_results.append({
                    "Model": m_name.upper(), "Dataset": "Train",
                    "RMSE_Tc": np.sqrt(mean_squared_error(y_train_tc, pred_train_tc)),
                    "MAE_Tc": mean_absolute_error(y_train_tc, pred_train_tc), "R2_Tc": r2_score(y_train_tc, pred_train_tc),
                    "RMSE_Pc": np.sqrt(mean_squared_error(y_train_pc, pred_train_pc)),
                    "MAE_Pc": mean_absolute_error(y_train_pc, pred_train_pc), "R2_Pc": r2_score(y_train_pc, pred_train_pc)})
                metrics_results.append({
                    "Model": m_name.upper(), "Dataset": "Test",
                    "RMSE_Tc": np.sqrt(mean_squared_error(y_test_tc, pred_test_tc)),
                    "MAE_Tc": mean_absolute_error(y_test_tc, pred_test_tc), "R2_Tc": r2_score(y_test_tc, pred_test_tc),
                    "RMSE_Pc": np.sqrt(mean_squared_error(y_test_pc, pred_test_pc)),
                    "MAE_Pc": mean_absolute_error(y_test_pc, pred_test_pc), "R2_Pc": r2_score(y_test_pc, pred_test_pc)})

            metrics_df = pd.DataFrame(metrics_results)
            st.success("Models training completed successfully!")
            
            st.subheader("📋 Model Performance Comparison (Metrics Summary)")
            st.dataframe(metrics_df, use_container_width=True)
            metrics_df.to_csv("models_output_plots/models_performance_metrics.csv", index=False)
            st.info("Results and chart files have been saved successfully to local directories.")
else:
    st.info("Please verify the file path in the sidebar and click the Start Training button.")

2026-11-05 20:05:59.496 
  command:

    streamlit run c:\Users\hp\.anaconda\Lib\site-packages\ipykernel_launcher.py [ARGUMENTS]
2026-11-05 20:05:59.500 Session state does not function when running a script without `streamlit run`
